# 기초 B1 · 확률과 분포

이 노트북은 구글 **Colab**에서 바로 실행됩니다. 위에서부터 각 셀을 **Shift+Enter** 로 실행하세요. 설치는 없고, 구글 계정만 있으면 됩니다.

📖 본문 학습 페이지: [기초 B1 · 확률과 분포](https://grow.minds.kr/textbooks/css-methods/causal/book/b1-확률과-분포.html)

## 1. 준비

In [ ]:
# 이 책의 데이터·코드를 코랩으로 내려받습니다(처음 한 번, 수 초).
!git clone -q https://github.com/dataminds/css-methods-causal-code.git
%cd css-methods-causal-code

In [ ]:
import pandas as pd, numpy as np
from scipy import stats

def load(name, clean=True):
    df = pd.read_csv(f"data/journey_{name}.csv")
    return df[df.attn_1 == 1] if clean and "attn_1" in df else df

def ols(y, X):                      # 절편 포함 최소제곱 → (계수, 표준오차, p, R^2)
    y = np.asarray(y, float)
    X1 = np.column_stack([np.ones(len(y))] + [np.asarray(x, float) for x in X])
    b, *_ = np.linalg.lstsq(X1, y, rcond=None)
    resid = y - X1 @ b
    n, k = X1.shape
    se = np.sqrt(np.diag(resid @ resid / (n - k) * np.linalg.inv(X1.T @ X1)))
    p = 2 * stats.t.sf(np.abs(b / se), n - k)
    r2 = 1 - (resid @ resid) / ((y - y.mean()) @ (y - y.mean()))
    return b, se, p, r2

def cohen_d(a, b):
    sp = np.sqrt(((len(a)-1)*a.std(ddof=1)**2 + (len(b)-1)*b.std(ddof=1)**2) / (len(a)+len(b)-2))
    return (a.mean() - b.mean()) / sp

def cronbach(items):
    items = np.asarray(items, float); k = items.shape[1]
    return k/(k-1) * (1 - items.var(axis=0, ddof=1).sum() / items.sum(axis=1).var(ddof=1))

print("준비 끝. 데이터와 도우미 함수를 불러왔습니다.")


## 2. 큰 수의 법칙
동전을 몇 번 던져야 0.5에 가까워지나. 한 번 한 번은 못 맞히는데 많이 모으면 맞는다는 것이 확률의 출발입니다.

In [ ]:
g = np.random.default_rng(73)
for n in (10, 100, 1000, 10000):
    print(n, round(g.integers(0, 2, n).mean(), 3))
# 10 0.5 / 100 0.42 / 1000 0.497 / 10000 0.499

## 3. 생김새가 다른 세 분포
평균이 같아도 분포는 다릅니다. 지수분포의 평균과 중앙값이 얼마나 벌어지는지 보세요.

In [ ]:
g = np.random.default_rng(73)
n = 5000
uni = g.uniform(0, 10, n)
nor = g.normal(5, 1.5, n)
exp_ = g.exponential(5, n)
for lab, v in [("균등", uni), ("정규", nor), ("지수", exp_)]:
    print(lab, round(v.mean(), 2), round(v.std(ddof=1), 2),
          round(float(np.median(v)), 2), round(float(stats.skew(v)), 2))
# 균등 5.01 2.89 5.01 0.0 / 정규 5.01 1.5 5.01 0.02 / 지수 4.95 4.92 3.4 1.91

## 4. 68·95·99.7은 어림이다
정규분포에서만 성립하고, 그마저 소수 셋째 자리에서 다릅니다.

In [ ]:
g = np.random.default_rng(73)
z = g.normal(0, 1, 100000)
for k in (1, 2, 3):
    print(k, round(float(np.mean(np.abs(z) <= k)), 3))
# 1 0.681 / 2 0.954 / 3 0.997

## 5. ⭐⭐ 이 단위의 심장
자료의 분포와 **통계량의 분포**는 다른 것입니다. 종 모양이 되는 것은 자료가 아니라 통계량입니다.

In [ ]:
svy = load("svy")
pop = svy.mil.values
g = np.random.default_rng(73)
means = np.array([g.choice(pop, 30, replace=False).mean() for _ in range(2000)])
print(round(pop.std(ddof=1), 3), round(float(stats.skew(pop)), 3))    # 자료 566명
print(round(means.std(ddof=1), 3), round(float(stats.skew(means)), 3))  # 30명 평균 2,000개
print(round(pop.std(ddof=1) / np.sqrt(30), 3))                        # 공식 SD/sqrt(30)
# 1.252 -0.306 / 0.222 -0.059 / 0.229
# 흩어짐이 다섯 배 줄고 비뚤어짐도 펴진다. 그리고 공식이 시뮬레이션을 맞힌다.

## 6. *t*분포가 정규로 수렴한다
표본이 작을수록 꼬리가 두껍습니다. 자유도가 커지면 1.96으로 갑니다.

In [ ]:
for df in (5, 30, 200):
    print(df, round(stats.t.ppf(.975, df), 3))
print(round(stats.norm.ppf(.975), 3))
# 5 2.571 / 30 2.042 / 200 1.972 / 1.96

## 4. 직접 바꿔 보기
위 셀의 숫자(씨앗 73, 표본 크기, 제외 기준 등)를 바꿔 다시 실행해 보세요. 결과가 어떻게 달라지나요?

> **검증 로그(부록 B)**: 무엇을 바꿨고, 무엇이 나왔고, 예상과 같았는지 한 문단으로 적어 두세요. 실행이 아니라 검증이 이 책의 핵심입니다.